In [2]:
# Install OpenAI Whisper with a stable CPU PyTorch build on Windows.
%pip uninstall -y whisper torch
%pip install --upgrade openai-whisper
%pip install --upgrade torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Note: you may need to restart the kernel to use updated packages.



  Using cached torch-2.12.1-cp311-cp311-win_amd64.whl (123.0 MB)
Looking in indexes: https://download.pytorch.org/whl/cpu
                                              0.0/205.5 MB ? eta -:--:--
                                              0.1/205.5 MB 2.0 MB/s eta 0:01:44
                                              0.1/205.5 MB 2.0 MB/s eta 0:01:44
                                              0.1/205.5 MB 2.0 MB/s eta 0:01:44
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:04
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
                                              0.2/205.5 MB 1.1 MB/s eta 0:03:14
    

In [11]:
from pathlib import Path
import whisper

audio_path = Path("audio.m4a")

if not audio_path.exists():
    print(f"Audio file not found: {audio_path.resolve()}")
else:
    # Use a stronger multilingual model for better Tamil -> English translation.
    model = whisper.load_model("medium")

    # Transcribe in source language (Tamil expected)
    transcription = model.transcribe(
        str(audio_path),
        task="transcribe",
        language="ta",
        fp16=False,
        temperature=0,
        condition_on_previous_text=False,
        no_speech_threshold=0.2,
    )
    tamil_text = transcription["text"].strip()
    print("Tamil Text:", tamil_text)

    # Translate speech to English directly with Whisper
    translation = model.transcribe(
        str(audio_path),
        task="translate",
        language="ta",
        fp16=False,
        temperature=0,
        condition_on_previous_text=False,
        no_speech_threshold=0.2,
        beam_size=5,
        best_of=5,
    )
    english_text = translation["text"].strip()

    # Retry once with a prompt if the first translation is empty.
    if not english_text:
        translation = model.transcribe(
            str(audio_path),
            task="translate",
            language="ta",
            fp16=False,
            temperature=0,
            condition_on_previous_text=False,
            no_speech_threshold=0.2,
            beam_size=5,
            best_of=5,
            initial_prompt="Translate the spoken Tamil clearly into English.",
        )
        english_text = translation["text"].strip()

    print("English Translation:", english_text if english_text else "[No translation produced]")

100%|█████████████████████████████████████| 1.42G/1.42G [03:06<00:00, 8.18MiB/s]


Tamil Text: யபா இந்தனெட்டு பில்லு கட்டியாச்சு வந்துட்டுச்சானு பாரு
English Translation: internet bill has been cut see if they have come


In [1]:
%pip install -q sounddevice scipy


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Live microphone streaming -> Whisper English translation (press Stop/Interrupt to end)

import queue
import tempfile
from pathlib import Path

import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write as wav_write
import whisper

sample_rate = 16000
channels = 1
chunk_seconds = 5  # Translate every 5 seconds
source_language = "ta"  # Set source language code

# Reuse already loaded model if available; otherwise load one.
if "model" not in globals() or model is None:
    model = whisper.load_model("medium")

print("Starting microphone stream. Speak now...")
print("Stop the cell to end streaming.")

audio_queue = queue.Queue()


def audio_callback(indata, frames, time_info, status):
    if status:
        print(status)
    audio_queue.put(indata.copy())


buffer = np.empty((0, channels), dtype=np.float32)

with sd.InputStream(
    samplerate=sample_rate,
    channels=channels,
    dtype="float32",
    callback=audio_callback,
):
    try:
        while True:
            # Collect audio frames from callback queue
            buffer = np.vstack([buffer, audio_queue.get()])

            if len(buffer) >= sample_rate * chunk_seconds:
                chunk = buffer[: sample_rate * chunk_seconds]
                buffer = buffer[sample_rate * chunk_seconds :]

                # Convert float32 [-1, 1] to int16 WAV for Whisper input
                chunk_int16 = np.int16(np.clip(chunk, -1.0, 1.0) * 32767)

                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_wav:
                    wav_path = Path(temp_wav.name)

                wav_write(wav_path, sample_rate, chunk_int16)

                try:
                    translation = model.transcribe(
                        str(wav_path),
                        task="translate",
                        language=source_language,
                        fp16=False,
                        temperature=0,
                        condition_on_previous_text=False,
                        no_speech_threshold=0.2,
                        beam_size=5,
                    )
                    english_text = translation.get("text", "").strip()
                    if english_text:
                        print("English:", english_text)
                finally:
                    wav_path.unlink(missing_ok=True)

    except KeyboardInterrupt:
        print("Streaming stopped.")

Starting microphone stream. Speak now...
Stop the cell to end streaming.
Streaming stopped.
